# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reetuparabat/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:


import pandas as pd
import numpy as np
from pathlib import Path

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
]
csv_path = next((p for p in candidates if Path(p).exists()), None)
if csv_path is None:
    matches = list(Path(".").rglob("content_refresh_anonymized.csv"))
    csv_path = str(matches[0]) if matches else None
if csv_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found — did the clone cell above run?")

print(f"loading: {csv_path}")
df = pd.read_csv(csv_path)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# ---- Signal check 1: CTR vs position (the signal behind FlyRank's CTR-fix logic) ----
sig1 = df.groupby('position_tier')['ctr'].agg(n='count', mean_ctr='mean', median_ctr='median') \
         .reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
print("SIGNAL 1 — CTR by position_tier (best position -> worst position):")
print(sig1)
print()

# ---- Signal check 2: staleness (the signal behind FlyRank's refresh flags) ----
sig2 = df.groupby('freshness_tier')['is_declining_label'].agg(n='count', decline_rate='mean') \
         .reindex(['0-30', '31-90', '91-180', '181+'])
print("SIGNAL 2 — decline rate by freshness_tier (fresh -> stale):")
print(sig2)

"""
1) My rule, in plain words, and the two signals it leans on

Signal 1 — CTR vs position (behind FlyRank's CTR-fix logic). Verdict: CONFIRMED.
  Mean CTR falls cleanly and monotonically as position gets worse: top_3=1.484%,
  page_1=0.652%, striking=0.323%, page_3_5=0.222%, deep=0.150%
  (n=2,321 / 11,814 / 7,304 / 7,242 / 1,319). Median CTR tells a sharper story:
  top_3 and deep both have a MEDIAN of 0.00% -- more than half of pages in the
  best and worst tiers get essentially zero clicks -- while page_1's median
  (0.16) is the only one clearly above zero. The mean relationship holds and is
  real, but it's driven by a right-skewed CTR distribution, which matters for
  how I benchmark "underperforming" below.

Signal 2 — staleness (behind FlyRank's refresh flags). Verdict: MIXED.
  Decline rate by freshness_tier: 0-30=51.1% (n=20,480), 31-90=58.9% (n=175),
  91-180=61.1% (n=9,171), 181+=47.1% (n=174). It rises from "just updated" to
  "91-180 days stale" as expected, but the STALEST bucket (181+) has the LOWEST
  decline rate of all four -- the opposite of a naive "older = more likely
  declining" prediction. That bucket is tiny (n=174) so I don't fully trust it,
  but the pattern isn't clean enough to bake "days_since_update >= X" into my
  score as a simple multiplier. This negative result is why my rule below does
  NOT use raw staleness as a gate or multiplier.

My rule, in plain words: "A page that's genuinely visible in search AND whose
click-through rate sits well below what pages at its own position tier
typically get is worth reviewing first -- it's ranking fine but something
(title, meta description, snippet) is stopping people from clicking."

Reason code (ONE, applied to every scored row): ctr_underperforming_for_position
Action label: review_for_ctr_fix
"""

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 144 (delta 52), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.85 MiB | 14.04 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
content_refresh_anonymized.csv
loading: data/raw/content_refresh_anonymized.csv
SIGNAL 1 — CTR by position_tier (best position -> worst position):
                   n  mean_ctr  median_ctr
position_tier                             
top_3           2321  1.483611        0.00
page_1         11814  0.652467        0.16
striking        7304  0.323239        0.11
page_3_5        7242  0.222484        0.03
deep            1319  0.150212        0.00

SIGNAL 2 — decline rate by freshness_tier (fresh -> stale):
                    n  decline_rate
freshness_tier           

'\n1) My rule, in plain words, and the two signals it leans on\n\nSignal 1 — CTR vs position (behind FlyRank\'s CTR-fix logic). Verdict: CONFIRMED.\n  Mean CTR falls cleanly and monotonically as position gets worse: top_3=1.484%,\n  page_1=0.652%, striking=0.323%, page_3_5=0.222%, deep=0.150%\n  (n=2,321 / 11,814 / 7,304 / 7,242 / 1,319). Median CTR tells a sharper story:\n  top_3 and deep both have a MEDIAN of 0.00% -- more than half of pages in the\n  best and worst tiers get essentially zero clicks -- while page_1\'s median\n  (0.16) is the only one clearly above zero. The mean relationship holds and is\n  real, but it\'s driven by a right-skewed CTR distribution, which matters for\n  how I benchmark "underperforming" below.\n\nSignal 2 — staleness (behind FlyRank\'s refresh flags). Verdict: MIXED.\n  Decline rate by freshness_tier: 0-30=51.1% (n=20,480), 31-90=58.9% (n=175),\n  91-180=61.1% (n=9,171), 181+=47.1% (n=174). It rises from "just updated" to\n  "91-180 days stale" as exp

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import os

df['visible'] = (df['impressions_90d'] >= 500).astype(int)

benchmark_ctr = df.groupby('position_tier')['ctr'].mean().to_dict()
df['benchmark_ctr'] = df['position_tier'].map(benchmark_ctr)

df['ctr_gap'] = (df['benchmark_ctr'] - df['ctr']).clip(lower=0)

df['score'] = df['visible'] * df['ctr_gap'] * df['impressions_90d'] / 100
df['reason_code'] = 'ctr_underperforming_for_position'
df['action'] = 'review_for_ctr_fix'

queue = df[df['score'] > 0].sort_values('score', ascending=False).reset_index(drop=True)

print(f"Total rows: {len(df):,}")
print(f"Excluded by visibility gate (impressions_90d < 500): {(df['visible']==0).sum():,}")
print(f"Ranked queue (score > 0): {len(queue):,} rows")

os.makedirs("work/outputs", exist_ok=True)
out_cols = ['content_id', 'client_id', 'position_tier', 'avg_position', 'ctr',
            'benchmark_ctr', 'impressions_90d', 'score', 'reason_code', 'action']
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("written: work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)

Total rows: 30,000
Excluded by visibility gate (impressions_90d < 500): 13,274
Ranked queue (score > 0): 13,698 rows
written: work/outputs/baseline_action_score.csv


,content_id,client_id,position_tier,avg_position,ctr,benchmark_ctr,impressions_90d,score,reason_code,action
0,content_8c19996aa890,client_4e07408562,top_3,2.5,0.15,1.483611,509252,6791.438208,ctr_underperforming_for_position,review_for_ctr_fix
1,content_4c36c775b818,client_4e07408562,top_3,2.3,0.41,1.483611,463103,4971.922493,ctr_underperforming_for_position,review_for_ctr_fix
2,content_8451fc6f034d,client_d029fa3a95,top_3,2.3,0.03,1.483611,272144,3955.913794,ctr_underperforming_for_position,review_for_ctr_fix
3,content_5fe46e04994d,client_4e07408562,page_1,4.2,0.14,0.652467,517715,2653.116277,ctr_underperforming_for_position,review_for_ctr_fix
4,content_44e481c8f55b,client_19581e27de,top_3,1.4,0.65,1.483611,312694,2606.650057,ctr_underperforming_for_position,review_for_ctr_fix
5,content_e12868d1f396,client_4e07408562,top_3,2.9,0.07,1.483611,149712,2116.344571,ctr_underperforming_for_position,review_for_ctr_fix
6,content_aaef01a50def,client_19581e27de,page_1,5.4,0.25,0.652467,517109,2081.190830,ctr_underperforming_for_position,review_for_ctr_fix
7,content_9532f197bbc8,client_4e07408562,top_3,2.0,0.87,1.483611,309192,1897.234616,ctr_underperforming_for_position,review_for_ctr_fix
8,content_4a6607efcb46,client_6208ef0f77,top_3,2.2,0.01,1.483611,128068,1887.223511,ctr_underperforming_for_position,review_for_ctr_fix
9,content_36ff89c8214e,client_19581e27de,page_1,7.3,0.05,0.652467,295097,1777.860760,ctr_underperforming_for_position,review_for_ctr_fix


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
"""
3) Top-10 review — action, why it's there, what would make it wrong

1. content_8c19996aa890 (client_4e07408562) — review_for_ctr_fix. top_3 (2.5),
   CTR 0.15% vs 1.484% benchmark, 509,252 impressions -- largest score in the
   whole queue.
   Wrong if: branded/navigational query where users search-then-type the URL
   directly instead of clicking the result -- low CTR would be expected, not
   a fixable snippet problem.

2. content_4c36c775b818 (client_4e07408562) — review_for_ctr_fix. top_3 (2.3),
   CTR 0.41% vs 1.484% benchmark, 463,103 impressions.
   Wrong if: title/meta was already changed recently and the 90-day CTR average
   still reflects the old snippet.

3. content_8451fc6f034d (client_d029fa3a95) — review_for_ctr_fix. top_3 (2.3),
   CTR 0.03% (near-zero) vs 1.484% benchmark, 272,144 impressions.
   Wrong if: this page is being impression-counted for a query it doesn't
   actually answer -- a content-to-query mismatch, not a snippet problem.

4. content_5fe46e04994d (client_4e07408562) — review_for_ctr_fix. page_1 (4.2),
   CTR 0.14% vs 0.652% benchmark, 517,715 impressions -- highest-volume row in
   the queue.
   Wrong if: impression volume this high for a single page is itself unusual --
   worth ruling out a tracking/dedup artifact before trusting the CTR gap.

5. content_44e481c8f55b (client_19581e27de) — review_for_ctr_fix. top_3 (1.4,
   near-best position), CTR 0.65% vs 1.484% benchmark, 312,694 impressions.
   Wrong if: transactional search intent naturally produces lower CTR
   (comparison shopping before clicking) -- my benchmark doesn't split by
   intent, so part of this gap may be intent mix, not a snippet issue.

6. content_e12868d1f396 (client_4e07408562) — review_for_ctr_fix. top_3 (2.9),
   CTR 0.07% vs 1.484% benchmark, 149,712 impressions.
   Wrong if: this page's update is only ~7 days old -- any fix may already be
   live and just hasn't accumulated enough fresh impressions to move the
   90-day CTR average yet.

7. content_aaef01a50def (client_19581e27de) — review_for_ctr_fix. page_1 (5.4),
   CTR 0.25% vs 0.652% benchmark, 517,109 impressions.
   Wrong if: this is the 2nd of 3 top-10 rows from the same client -- if the
   account has a systematic tracking/tagging quirk, this is a client-level
   measurement issue, not a page-level one.

8. content_9532f197bbc8 (client_4e07408562) — review_for_ctr_fix. top_3 (2.0),
   CTR 0.87% vs 1.484% benchmark, 309,192 impressions.
   Wrong if: 0.87% isn't obviously bad in absolute terms -- see section 4, the
   mean benchmark here is pulled up by outliers; under a median benchmark this
   row scores 0 and drops out of the queue entirely.

9. content_4a6607efcb46 (client_6208ef0f77) — review_for_ctr_fix. top_3 (2.2),
   CTR 0.01% vs 1.484% benchmark, 128,068 impressions.
   Wrong if: CTR this close to zero at top_3 suggests the result may not be
   rendering normally (missing snippet/thumbnail) -- a technical/indexing
   issue, not something a content review would fix.

10. content_36ff89c8214e (client_19581e27de) — review_for_ctr_fix. page_1 (7.3),
    CTR 0.05% vs 0.652% benchmark, 295,097 impressions.
    Wrong if: 3rd of 3 top-10 rows from this client -- see #7's caveat; also
    this exact row reappears near the top of the median-benchmark version in
    section 4, which is one of my more robust picks across both benchmarks.
"""

"\n3) Top-10 review — action, why it's there, what would make it wrong\n\n1. content_8c19996aa890 (client_4e07408562) — review_for_ctr_fix. top_3 (2.5),\n   CTR 0.15% vs 1.484% benchmark, 509,252 impressions -- largest score in the\n   whole queue.\n   Wrong if: branded/navigational query where users search-then-type the URL\n   directly instead of clicking the result -- low CTR would be expected, not\n   a fixable snippet problem.\n\n2. content_4c36c775b818 (client_4e07408562) — review_for_ctr_fix. top_3 (2.3),\n   CTR 0.41% vs 1.484% benchmark, 463,103 impressions.\n   Wrong if: title/meta was already changed recently and the 90-day CTR average\n   still reflects the old snippet.\n\n3. content_8451fc6f034d (client_d029fa3a95) — review_for_ctr_fix. top_3 (2.3),\n   CTR 0.03% (near-zero) vs 1.484% benchmark, 272,144 impressions.\n   Wrong if: this page is being impression-counted for a query it doesn't\n   actually answer -- a content-to-query mismatch, not a snippet problem.\n\n4. con

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
median_benchmark_ctr = df.groupby('position_tier')['ctr'].median().to_dict()
df['benchmark_ctr_median'] = df['position_tier'].map(median_benchmark_ctr)
df['ctr_gap_median'] = (df['benchmark_ctr_median'] - df['ctr']).clip(lower=0)
df['score_median'] = df['visible'] * df['ctr_gap_median'] * df['impressions_90d'] / 100

top10_mean = set(queue.head(10)['content_id'])
top10_median = set(df.sort_values('score_median', ascending=False).head(10)['content_id'])
overlap = top10_mean & top10_median

print("median CTR benchmark by tier:", median_benchmark_ctr)
print(f"rows scored under mean benchmark:   {(df['score'] > 0).sum():,}")
print(f"rows scored under median benchmark: {(df['score_median'] > 0).sum():,}")
print(f"top-10 overlap between mean and median benchmark: {len(overlap)}/10")
print("shared row(s):", overlap)

"""
4) Weak picks, robustness check, and leakage check

Robustness check — mean vs median benchmark:
  Switching the benchmark from mean to median CTR only agrees on 1 of the top
  10 rows, and cuts the total scored queue from 13,698 rows to 4,973. The
  reason is structural, not noise: top_3 and deep tiers both have a MEDIAN
  CTR of exactly 0.00% (signal 1), so under a median benchmark, gap = max(0,
  0 - ctr) is always 0 for those tiers -- a median-based rule can NEVER flag a
  top_3 or deep page as underperforming, no matter how bad its CTR is. That
  silently exempts my best- and worst-position pages from review entirely,
  which is worse than the mean's outlier-sensitivity. I'm keeping the mean
  benchmark, but flagging this as a known weakness: the mean is pulled upward
  by a right-skewed CTR distribution, so some "underperforming" rows (like
  #8 above) may not look bad against a more typical page in their tier.

Weakest pick: content_9532f197bbc8 (#8 in section 3). It only scores high
because the mean benchmark for top_3 (1.484%) is skewed upward by a handful
of very-high-CTR pages; under the median benchmark it scores 0 and disappears
from the queue completely.

Concentration problem found during review: 5 of the top 10, and 3 of the
remaining 5, come from just two clients (client_4e07408562 and
client_19581e27de) out of 32 distinct clients in the dataset. Not leakage, but
a real fairness/coverage risk: this baseline could keep re-flagging the same
large clients' pages while smaller clients never surface, simply because they
have more high-volume pages -- not because their content is worse.

Leakage check: score is built from ctr, position_tier, avg_position, and
impressions_90d -- all trailing-90-day observed signals. None touch
trend_direction, trend_pct, or is_declining_label (the label source), and none
are FlyRank product-decision outputs (health_score, priority_score,
action_type aren't in this dataset). No future window is used anywhere.
"""

median CTR benchmark by tier: {'deep': 0.0, 'page_1': 0.16, 'page_3_5': 0.03, 'striking': 0.11, 'top_3': 0.0}
rows scored under mean benchmark:   13,698
rows scored under median benchmark: 4,973
top-10 overlap between mean and median benchmark: 1/10
shared row(s): {'content_36ff89c8214e'}


'\n4) Weak picks, robustness check, and leakage check\n\nRobustness check — mean vs median benchmark:\n  Switching the benchmark from mean to median CTR only agrees on 1 of the top\n  10 rows, and cuts the total scored queue from 13,698 rows to 4,973. The\n  reason is structural, not noise: top_3 and deep tiers both have a MEDIAN\n  CTR of exactly 0.00% (signal 1), so under a median benchmark, gap = max(0,\n  0 - ctr) is always 0 for those tiers -- a median-based rule can NEVER flag a\n  top_3 or deep page as underperforming, no matter how bad its CTR is. That\n  silently exempts my best- and worst-position pages from review entirely,\n  which is worse than the mean\'s outlier-sensitivity. I\'m keeping the mean\n  benchmark, but flagging this as a known weakness: the mean is pulled upward\n  by a right-skewed CTR distribution, so some "underperforming" rows (like\n  #8 above) may not look bad against a more typical page in their tier.\n\nWeakest pick: content_9532f197bbc8 (#8 in sectio

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.